# AffectScore — Gate Scripts

Run the three gate checks in order. All three must pass before starting LoRA training.

**Requirements**
- Colab Pro+ (A100 GPU recommended)
- `affectscore-colab.zip` uploaded to `MyDrive/affectscore/affectscore-colab.zip`
- Hugging Face token with read access (needed to download the ACE-Step base model)
- Training WAVs on Drive — downloaded via the Zenodo cell below (Gate 3 only)

In [ ]:
from google.colab import drive, userdata
import os, subprocess

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/affectscore"
REPO  = "/content/affectscore"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/LeeTgk/affectscore.git", REPO], check=True)

print(f"Repo ready: {REPO}")


In [ ]:
!bash /content/affectscore/training/colab_setup.sh


## Download training data

Gate 3 requires the preprocessed WAV files to calibrate the CLAP loss weight λ.
Gates 1 and 2 do not need the WAV files — skip to **Start server** if you only need those.

In [ ]:
import os, subprocess, shutil, tarfile, glob

ZENODO_RECORD = "21830658"   # https://zenodo.org/record/21830658
PREPROCESSED  = f"{DRIVE}/preprocessed_unfiltered"

if not os.path.isdir(PREPROCESSED):
    print("Downloading dataset from Zenodo...")
    subprocess.run(["pip", "install", "zenodo-get", "-q"], check=True)
    subprocess.run(["zenodo_get", ZENODO_RECORD, "-o", DRIVE], check=True)

    # Extract archive if Zenodo record ships a tar.gz
    for archive in glob.glob(f"{DRIVE}/*.tar.gz"):
        print(f"Extracting {archive}...")
        with tarfile.open(archive) as t:
            t.extractall(DRIVE)
    print("Download complete.")
else:
    print(f"Dataset already present: {PREPROCESSED}")

# Copy dataset manifests into repo data/ so scripts can find them
os.makedirs(f"{REPO}/data", exist_ok=True)
for fname in ["training_set_clean_clap.json", "held_out_set.json"]:
    _src = f"{DRIVE}/{fname}"
    _dst = f"{REPO}/data/{fname}"
    if os.path.exists(_src) and not os.path.exists(_dst):
        shutil.copy2(_src, _dst)
        print(f"Copied {fname} -> {REPO}/data/")



## Start server

The server loads the ACE-Step base model (~2 min on A100).

In [ ]:
import subprocess, time, requests, os

subprocess.run(["pkill", "-f", "affectscore_server.py"], capture_output=True)
time.sleep(2)

proc = subprocess.Popen(
    ["python", "server/affectscore_server.py", "--port", "8321", "--device-id", "0"],
    cwd=REPO,
    stdout=open("/tmp/server.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Server PID: {proc.pid} — polling for readiness...")

for i in range(150):
    time.sleep(5)
    try:
        r = requests.get("http://127.0.0.1:8321/health", timeout=2)
        if r.json().get("status") == "ok":
            print(f"Server ready ({(i+1)*5}s):", r.json()); break
    except:
        pass
    if (i+1) % 12 == 0:
        print(f"  Still loading... {(i+1)*5}s")
else:
    print("ERROR: server timed out — check /tmp/server.log")
    os.system("tail -30 /tmp/server.log")

## Gate 1 — Real-Time Factor

**Target:** < 2 s end-to-end for a 4 s chunk at 8 inference steps (RTF < 0.5).

Client-side timing: HTTP round-trip + generation + WAV encode + file write.

In [ ]:
%cd /content/affectscore
!python training/gates/gate1_rtf.py --n-trials 20

In [ ]:
!cat training/gates/gate1_results.json


## Gate 2 — Attention Entropy

**Target:** cross-attention entropy varies > 0.02 bits across five widely-spaced V-A inputs,
confirming that the conditioning signal reaches the attention layers.

In [ ]:
%cd /content/affectscore
!python training/gates/gate2_attention.py

In [ ]:
!cat training/gates/gate2_results.json


## Gate 3 — CLAP Lambda

Calibrates the CLAP auxiliary loss weight λ on a 200-clip validation sample.
Requires the training WAVs in Drive (downloaded above).

In [ ]:
%cd /content/affectscore
!python training/gates/gate3_lambda.py \
  --audio-dir /content/drive/MyDrive/affectscore/preprocessed_unfiltered

In [ ]:
!cat training/gates/gate3_results.json
